# 13 Candidate Sentence Mode — NLP Sentence Builder

This notebook builds the first NLP layer for **Be My Ear / Be My Voice**.

The goal is to turn uncertain WLASL2000 outputs into a useful sentence:

```text
Top-5 candidate signs from each sign event
→ score possible sign sequences
→ choose the most meaningful sequence
→ build natural English sentence
```

This is not a huge NLP model yet. It is a controlled, explainable first NLP module that can later be connected to your app.

# 1. Import libraries

In [ ]:
from pathlib import Path
from itertools import product
import json
import math
import re

import numpy as np
import pandas as pd

# 2. Project paths

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

NLP_REPORT_DIR = PROJECT_ROOT / "reports" / "nlp"
NLP_REPORT_DIR.mkdir(parents=True, exist_ok=True)

NLP_OUTPUT_DIR = PROJECT_ROOT / "src" / "nlp"
NLP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("NLP report dir:", NLP_REPORT_DIR)
print("NLP source dir:", NLP_OUTPUT_DIR)

# 3. Example candidate sign events

In [ ]:
candidate_events = [
    {
        "event_id": 1,
        "time": 0.8,
        "status": "uncertain",
        "top_k": [
            {"gloss": "me", "probability": 0.30},
            {"gloss": "I", "probability": 0.25},
            {"gloss": "my", "probability": 0.12},
            {"gloss": "mine", "probability": 0.08},
            {"gloss": "myself", "probability": 0.05},
        ],
    },
    {
        "event_id": 2,
        "time": 1.6,
        "status": "locked",
        "top_k": [
            {"gloss": "want", "probability": 0.34},
            {"gloss": "need", "probability": 0.22},
            {"gloss": "like", "probability": 0.12},
            {"gloss": "ask", "probability": 0.06},
            {"gloss": "help", "probability": 0.05},
        ],
    },
    {
        "event_id": 3,
        "time": 2.4,
        "status": "uncertain",
        "top_k": [
            {"gloss": "water", "probability": 0.28},
            {"gloss": "drink", "probability": 0.20},
            {"gloss": "cup", "probability": 0.12},
            {"gloss": "milk", "probability": 0.09},
            {"gloss": "coffee", "probability": 0.07},
        ],
    },
]

print(json.dumps(candidate_events, indent=4))

# 4. Gloss normalisation and categories

In [ ]:
GLOSS_NORMALISATION = {
    "me": "I", "i": "I", "I": "I", "my": "my", "mine": "my", "myself": "I",
    "you": "you", "your": "your",
    "want": "want", "need": "need", "like": "like", "ask": "ask", "help": "help",
    "water": "water", "drink": "drink", "cup": "cup", "milk": "milk", "coffee": "coffee",
    "hospital": "hospital", "doctor": "doctor", "pain": "pain", "stomach": "stomach",
    "abdomen": "stomach", "sick": "sick", "emergency": "emergency",
    "bathroom": "bathroom", "toilet": "bathroom",
    "food": "food", "eat": "eat", "hungry": "hungry",
    "home": "home", "go": "go", "school": "school", "work": "work",
    "yes": "yes", "no": "no", "please": "please",
    "thank": "thank you", "thanks": "thank you", "sorry": "sorry",
}

SEMANTIC_CATEGORY = {
    "I": "person", "you": "person", "my": "person",
    "want": "intent", "need": "intent", "like": "intent", "ask": "intent",
    "help": "help",
    "water": "object", "drink": "object", "cup": "object",
    "milk": "object", "coffee": "object", "food": "object",
    "eat": "action", "go": "action",
    "hospital": "place", "doctor": "person_medical",
    "bathroom": "place", "home": "place", "school": "place", "work": "place",
    "pain": "medical", "stomach": "body", "sick": "medical", "emergency": "emergency",
    "yes": "response", "no": "response",
    "please": "politeness", "thank you": "politeness", "sorry": "politeness",
}

def normalise_gloss(gloss):
    gloss = str(gloss).strip()
    return GLOSS_NORMALISATION.get(gloss.lower(), gloss)

def get_category(gloss):
    return SEMANTIC_CATEGORY.get(gloss, "unknown")

# 5. Generate candidate sequences

In [ ]:
def get_top_candidates(event, max_candidates_per_event=3):
    candidates = []
    for item in event["top_k"][:max_candidates_per_event]:
        normalised = normalise_gloss(item["gloss"])
        candidates.append({
            "raw_gloss": item["gloss"],
            "gloss": normalised,
            "probability": float(item["probability"]),
            "category": get_category(normalised),
            "event_id": event["event_id"],
            "status": event.get("status", "unknown"),
        })
    return candidates

def generate_candidate_sequences(events, max_candidates_per_event=3):
    candidate_lists = [get_top_candidates(e, max_candidates_per_event) for e in events]
    return [list(seq) for seq in product(*candidate_lists)]

sequences = generate_candidate_sequences(candidate_events, max_candidates_per_event=3)
print("Number of candidate sequences:", len(sequences))
print(sequences[0])

# 6. Score possible sign sequences

In [ ]:
def probability_score(sequence):
    probs = [max(item["probability"], 1e-6) for item in sequence]
    return sum(math.log(p) for p in probs) / max(len(probs), 1)

def status_bonus(sequence):
    bonus = 0.0
    for item in sequence:
        if item["status"] == "accepted":
            bonus += 0.25
        elif item["status"] == "locked":
            bonus += 0.18
        elif item["status"] == "uncertain":
            bonus += 0.05
    return bonus

def grammar_pattern_score(glosses, categories):
    score = 0.0

    if len(categories) >= 3:
        if categories[0] == "person" and categories[1] == "intent" and categories[2] in ["object", "place"]:
            score += 1.2

    if len(categories) >= 2:
        if categories[0] == "intent" and categories[1] in ["object", "place"]:
            score += 0.8

    if "pain" in glosses and ("stomach" in glosses or "body" in categories):
        score += 1.0

    if "help" in glosses and any(c in ["medical", "place", "body"] for c in categories):
        score += 0.9

    for i in range(len(categories) - 1):
        if glosses[i] == "go" and categories[i + 1] == "place":
            score += 0.8

    if "please" in glosses or "thank you" in glosses:
        score += 0.15

    return score

def semantic_usefulness_score(glosses, categories):
    score = 0.0
    useful = {"person", "intent", "object", "place", "medical", "body", "help", "action", "emergency"}
    score += sum(0.1 for c in categories if c in useful)

    if any(g in ["emergency", "help", "hospital", "doctor", "pain", "sick"] for g in glosses):
        score += 0.5

    return score

def score_sequence(sequence):
    glosses = [item["gloss"] for item in sequence]
    categories = [item["category"] for item in sequence]

    p = probability_score(sequence)
    g = grammar_pattern_score(glosses, categories)
    s = semantic_usefulness_score(glosses, categories)
    b = status_bonus(sequence)

    return {
        "glosses": glosses,
        "categories": categories,
        "probability_score": p,
        "grammar_score": g,
        "semantic_score": s,
        "status_bonus": b,
        "total_score": p + g + s + b,
    }

scored_sequences = [{"sequence": seq, **score_sequence(seq)} for seq in sequences]

scored_df = pd.DataFrame([
    {
        "glosses": " ".join(x["glosses"]),
        "categories": " ".join(x["categories"]),
        "probability_score": x["probability_score"],
        "grammar_score": x["grammar_score"],
        "semantic_score": x["semantic_score"],
        "status_bonus": x["status_bonus"],
        "total_score": x["total_score"],
    }
    for x in scored_sequences
]).sort_values("total_score", ascending=False)

display(scored_df.head(10))

# 7. Build natural sentence from selected glosses

In [ ]:
def remove_repeated_words(words):
    cleaned = []
    for word in words:
        if len(cleaned) == 0 or cleaned[-1] != word:
            cleaned.append(word)
    return cleaned

def build_natural_sentence(glosses):
    glosses = [normalise_gloss(g) for g in glosses]
    glosses = remove_repeated_words(glosses)
    gloss_set = set(glosses)

    if "emergency" in gloss_set:
        return "This is an emergency."

    if "help" in gloss_set and "hospital" in gloss_set:
        return "I need help. I need to go to the hospital."

    if "pain" in gloss_set and "stomach" in gloss_set:
        return "I have stomach pain."

    if "sick" in gloss_set and "doctor" in gloss_set:
        return "I am sick. I need a doctor."

    if "go" in gloss_set and "hospital" in gloss_set:
        return "I need to go to the hospital."

    if "bathroom" in gloss_set:
        if "need" in gloss_set or "want" in gloss_set:
            return "I need to use the bathroom."
        return "Bathroom."

    subject, intent, obj, politeness = None, None, None, None

    for g in glosses:
        c = get_category(g)
        if subject is None and c == "person":
            subject = g
        if intent is None and c == "intent":
            intent = g
        if obj is None and c in ["object", "place"]:
            obj = g
        if c == "politeness":
            politeness = g

    if subject is None and intent is not None:
        subject = "I"

    if subject is not None and intent is not None and obj is not None:
        sentence = f"{subject} {intent} {obj}"
        if politeness == "please":
            sentence += ", please"
        return sentence[0].upper() + sentence[1:] + "."

    if intent is not None and obj is not None:
        sentence = f"I {intent} {obj}"
        if politeness == "please":
            sentence += ", please"
        return sentence[0].upper() + sentence[1:] + "."

    if "hungry" in gloss_set:
        return "I am hungry."

    if "water" in gloss_set or "drink" in gloss_set:
        return "I want a drink."

    if "thank you" in gloss_set:
        return "Thank you."

    if "sorry" in gloss_set:
        return "I am sorry."

    if len(glosses) == 1:
        return glosses[0].capitalize() + "."

    sentence = " ".join(glosses)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence[0].upper() + sentence[1:] + "."

# 8. Candidate Sentence Mode function

In [ ]:
def sentence_confidence_label(best_item, scored_sequences):
    sorted_scores = sorted([item["total_score"] for item in scored_sequences], reverse=True)
    best = sorted_scores[0]
    second = sorted_scores[1] if len(sorted_scores) > 1 else best - 1.0
    gap = best - second

    if best_item["grammar_score"] >= 1.0 and gap >= 0.15:
        return "High", gap

    if best_item["grammar_score"] >= 0.8 or best_item["semantic_score"] >= 0.5:
        return "Medium", gap

    return "Low", gap

def candidate_sentence_mode(events, max_candidates_per_event=3):
    sequences = generate_candidate_sequences(events, max_candidates_per_event)
    scored = [{"sequence": seq, **score_sequence(seq)} for seq in sequences]

    best = max(scored, key=lambda x: x["total_score"])
    sentence = build_natural_sentence(best["glosses"])
    confidence, gap = sentence_confidence_label(best, scored)

    alternatives = sorted(scored, key=lambda x: x["total_score"], reverse=True)[:5]

    return {
        "sentence": sentence,
        "confidence": confidence,
        "score_gap": float(gap),
        "selected_glosses": best["glosses"],
        "selected_categories": best["categories"],
        "best_score": float(best["total_score"]),
        "alternatives": [
            {
                "glosses": alt["glosses"],
                "sentence": build_natural_sentence(alt["glosses"]),
                "score": float(alt["total_score"])
            }
            for alt in alternatives
        ]
    }

result = candidate_sentence_mode(candidate_events, max_candidates_per_event=3)
print(json.dumps(result, indent=4))

# 9. Test emergency example

In [ ]:
emergency_events = [
    {
        "event_id": 1,
        "time": 0.5,
        "status": "locked",
        "top_k": [
            {"gloss": "me", "probability": 0.25},
            {"gloss": "I", "probability": 0.20},
            {"gloss": "my", "probability": 0.10},
        ],
    },
    {
        "event_id": 2,
        "time": 1.2,
        "status": "uncertain",
        "top_k": [
            {"gloss": "pain", "probability": 0.22},
            {"gloss": "sick", "probability": 0.18},
            {"gloss": "help", "probability": 0.14},
        ],
    },
    {
        "event_id": 3,
        "time": 2.0,
        "status": "uncertain",
        "top_k": [
            {"gloss": "abdomen", "probability": 0.20},
            {"gloss": "stomach", "probability": 0.17},
            {"gloss": "hospital", "probability": 0.12},
        ],
    },
]

emergency_result = candidate_sentence_mode(emergency_events, max_candidates_per_event=3)
print(json.dumps(emergency_result, indent=4))

# 10. Save demo output

In [ ]:
demo_output = {
    "candidate_events": candidate_events,
    "candidate_sentence_result": result,
    "emergency_events": emergency_events,
    "emergency_sentence_result": emergency_result
}

output_file = NLP_REPORT_DIR / "candidate_sentence_mode_demo_output.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(demo_output, f, indent=4)

print("Saved:", output_file)

# 11. Export reusable NLP module

In [ ]:
module_code = '''
from itertools import product
import math
import re
import numpy as np

GLOSS_NORMALISATION = {
    "me": "I", "i": "I", "I": "I", "my": "my", "mine": "my", "myself": "I",
    "you": "you", "your": "your",
    "want": "want", "need": "need", "like": "like", "ask": "ask", "help": "help",
    "water": "water", "drink": "drink", "cup": "cup", "milk": "milk", "coffee": "coffee",
    "hospital": "hospital", "doctor": "doctor", "pain": "pain", "stomach": "stomach",
    "abdomen": "stomach", "sick": "sick", "emergency": "emergency",
    "bathroom": "bathroom", "toilet": "bathroom",
    "food": "food", "eat": "eat", "hungry": "hungry",
    "home": "home", "go": "go", "school": "school", "work": "work",
    "yes": "yes", "no": "no", "please": "please",
    "thank": "thank you", "thanks": "thank you", "sorry": "sorry",
}

SEMANTIC_CATEGORY = {
    "I": "person", "you": "person", "my": "person",
    "want": "intent", "need": "intent", "like": "intent", "ask": "intent",
    "help": "help",
    "water": "object", "drink": "object", "cup": "object",
    "milk": "object", "coffee": "object", "food": "object",
    "eat": "action", "go": "action",
    "hospital": "place", "doctor": "person_medical",
    "bathroom": "place", "home": "place", "school": "place", "work": "place",
    "pain": "medical", "stomach": "body", "sick": "medical", "emergency": "emergency",
    "yes": "response", "no": "response",
    "please": "politeness", "thank you": "politeness", "sorry": "politeness",
}

def normalise_gloss(gloss):
    gloss = str(gloss).strip()
    return GLOSS_NORMALISATION.get(gloss.lower(), gloss)

def get_category(gloss):
    return SEMANTIC_CATEGORY.get(gloss, "unknown")

def get_top_candidates(event, max_candidates_per_event=3):
    candidates = []
    for item in event["top_k"][:max_candidates_per_event]:
        normalised = normalise_gloss(item["gloss"])
        candidates.append({
            "raw_gloss": item["gloss"],
            "gloss": normalised,
            "probability": float(item["probability"]),
            "category": get_category(normalised),
            "event_id": event["event_id"],
            "status": event.get("status", "unknown"),
        })
    return candidates

def generate_candidate_sequences(events, max_candidates_per_event=3):
    candidate_lists = [get_top_candidates(e, max_candidates_per_event) for e in events]
    return [list(seq) for seq in product(*candidate_lists)]

def probability_score(sequence):
    probs = [max(item["probability"], 1e-6) for item in sequence]
    return sum(math.log(p) for p in probs) / max(len(probs), 1)

def status_bonus(sequence):
    bonus = 0.0
    for item in sequence:
        if item["status"] == "accepted":
            bonus += 0.25
        elif item["status"] == "locked":
            bonus += 0.18
        elif item["status"] == "uncertain":
            bonus += 0.05
    return bonus

def grammar_pattern_score(glosses, categories):
    score = 0.0
    if len(categories) >= 3:
        if categories[0] == "person" and categories[1] == "intent" and categories[2] in ["object", "place"]:
            score += 1.2
    if len(categories) >= 2:
        if categories[0] == "intent" and categories[1] in ["object", "place"]:
            score += 0.8
    if "pain" in glosses and ("stomach" in glosses or "body" in categories):
        score += 1.0
    if "help" in glosses and any(c in ["medical", "place", "body"] for c in categories):
        score += 0.9
    for i in range(len(categories) - 1):
        if glosses[i] == "go" and categories[i + 1] == "place":
            score += 0.8
    if "please" in glosses or "thank you" in glosses:
        score += 0.15
    return score

def semantic_usefulness_score(glosses, categories):
    score = 0.0
    useful = {"person", "intent", "object", "place", "medical", "body", "help", "action", "emergency"}
    score += sum(0.1 for c in categories if c in useful)
    if any(g in ["emergency", "help", "hospital", "doctor", "pain", "sick"] for g in glosses):
        score += 0.5
    return score

def score_sequence(sequence):
    glosses = [item["gloss"] for item in sequence]
    categories = [item["category"] for item in sequence]
    p = probability_score(sequence)
    g = grammar_pattern_score(glosses, categories)
    s = semantic_usefulness_score(glosses, categories)
    b = status_bonus(sequence)
    return {
        "glosses": glosses,
        "categories": categories,
        "probability_score": p,
        "grammar_score": g,
        "semantic_score": s,
        "status_bonus": b,
        "total_score": p + g + s + b,
    }

def remove_repeated_words(words):
    cleaned = []
    for word in words:
        if len(cleaned) == 0 or cleaned[-1] != word:
            cleaned.append(word)
    return cleaned

def build_natural_sentence(glosses):
    glosses = [normalise_gloss(g) for g in glosses]
    glosses = remove_repeated_words(glosses)
    gloss_set = set(glosses)

    if "emergency" in gloss_set:
        return "This is an emergency."
    if "help" in gloss_set and "hospital" in gloss_set:
        return "I need help. I need to go to the hospital."
    if "pain" in gloss_set and "stomach" in gloss_set:
        return "I have stomach pain."
    if "sick" in gloss_set and "doctor" in gloss_set:
        return "I am sick. I need a doctor."
    if "go" in gloss_set and "hospital" in gloss_set:
        return "I need to go to the hospital."
    if "bathroom" in gloss_set:
        if "need" in gloss_set or "want" in gloss_set:
            return "I need to use the bathroom."
        return "Bathroom."

    subject, intent, obj, politeness = None, None, None, None
    for g in glosses:
        c = get_category(g)
        if subject is None and c == "person":
            subject = g
        if intent is None and c == "intent":
            intent = g
        if obj is None and c in ["object", "place"]:
            obj = g
        if c == "politeness":
            politeness = g

    if subject is None and intent is not None:
        subject = "I"

    if subject is not None and intent is not None and obj is not None:
        sentence = f"{subject} {intent} {obj}"
        if politeness == "please":
            sentence += ", please"
        return sentence[0].upper() + sentence[1:] + "."

    if intent is not None and obj is not None:
        sentence = f"I {intent} {obj}"
        if politeness == "please":
            sentence += ", please"
        return sentence[0].upper() + sentence[1:] + "."

    if "hungry" in gloss_set:
        return "I am hungry."
    if "water" in gloss_set or "drink" in gloss_set:
        return "I want a drink."
    if "thank you" in gloss_set:
        return "Thank you."
    if "sorry" in gloss_set:
        return "I am sorry."
    if len(glosses) == 1:
        return glosses[0].capitalize() + "."

    sentence = " ".join(glosses)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence[0].upper() + sentence[1:] + "."

def sentence_confidence_label(best_item, scored_sequences):
    sorted_scores = sorted([item["total_score"] for item in scored_sequences], reverse=True)
    best = sorted_scores[0]
    second = sorted_scores[1] if len(sorted_scores) > 1 else best - 1.0
    gap = best - second
    if best_item["grammar_score"] >= 1.0 and gap >= 0.15:
        return "High", gap
    if best_item["grammar_score"] >= 0.8 or best_item["semantic_score"] >= 0.5:
        return "Medium", gap
    return "Low", gap

def candidate_sentence_mode(events, max_candidates_per_event=3):
    sequences = generate_candidate_sequences(events, max_candidates_per_event)
    scored = [{"sequence": seq, **score_sequence(seq)} for seq in sequences]
    best = max(scored, key=lambda x: x["total_score"])
    sentence = build_natural_sentence(best["glosses"])
    confidence, gap = sentence_confidence_label(best, scored)
    alternatives = sorted(scored, key=lambda x: x["total_score"], reverse=True)[:5]
    return {
        "sentence": sentence,
        "confidence": confidence,
        "score_gap": float(gap),
        "selected_glosses": best["glosses"],
        "selected_categories": best["categories"],
        "best_score": float(best["total_score"]),
        "alternatives": [
            {
                "glosses": alt["glosses"],
                "sentence": build_natural_sentence(alt["glosses"]),
                "score": float(alt["total_score"]),
            }
            for alt in alternatives
        ],
    }
'''

module_file = NLP_OUTPUT_DIR / "candidate_sentence_mode.py"

with open(module_file, "w", encoding="utf-8") as f:
    f.write(module_code)

print("Saved reusable NLP module:", module_file)

# 12. Next connection step

Later, your live inference notebook should collect detected sign events in this format:

```python
candidate_events = [
    {
        "event_id": 1,
        "time": 0.8,
        "status": "locked",
        "top_k": [
            {"gloss": "me", "probability": 0.30},
            {"gloss": "I", "probability": 0.25},
            {"gloss": "my", "probability": 0.12}
        ]
    }
]
```

Then the app can call:

```python
from src.nlp.candidate_sentence_mode import candidate_sentence_mode

result = candidate_sentence_mode(candidate_events)
print(result["sentence"])
```